# Color-based dart pipeline: inspect board preparation

Milestone 2 runs only the real board-localization and normalization stages. No dummy orientation, segmentation, tip, or scoring stages are needed. Choose a multi-dart photo below, then rerun from **Run preparation** when switching photos/settings.

Look for concentric bull/ring alignment, sector spokes matching boundaries, and perspective errors. Magenta lines are **nominal model geometry**, not detected physical wires. The detector's color-derived rotation is ambiguous by 36 degrees; no sector numbers are confirmed yet. The current normalized canvas ends at the outer double ring and excludes the printed-number ring, so the future orientation stage will need an extended view or source-image sampling.


## Kernel and rerunning after code changes

Select this repository's `.venv` Python kernel (created with
`uv sync --group dev`). After pulling code or changing stage/result
contracts, use **Restart Kernel**, then **Run All** from the top.
This also resolves stale imports such as `ColorAssignment` and old
analysis artifacts such as missing `color_model_names`. Rerunning
only a display cell cannot update an existing analysis result.
Autoreload picks up subsequent Python edits; rerun the
analysis cells to recompute results. Restart again for structural
class/dataclass changes rather than mixing old and new objects.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = next((p for p in (cwd, *cwd.parents) if (p / 'pyproject.toml').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open the notebook from this repository or its examples directory.')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from dartcv.color_dart_pipeline import prepare_board, format_timing_summary
from dartcv.color_dart_inspection import plot_board_preparation


In [ ]:
image_dir = repo_root / 'data' / 'images' / 'multi_dart'
image_paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
if not image_paths:
    raise FileNotFoundError(f'No photos in {image_dir}; restore the local DVC dataset first.')
for index, path in enumerate(image_paths):
    print(f'{index:2d}: {path.name}')


## Run preparation

Change `IMAGE_INDEX` and rerun this and subsequent cells. Images are loaded in BGR without resizing; display previews are reduced, while stored diagnostics retain their original resolution. Keep only the selected run: these 50-megapixel photos make diagnostic snapshots memory-intensive. No files are written.

Timings measure each stage's execution, including diagnostic work inside that stage, but exclude file I/O, snapshot-copy overhead, and rendering. Disable diagnostics for performance comparisons; a successful run alone does not establish geometric accuracy.


In [ ]:
IMAGE_INDEX = 0
NORMALIZED_SIZE_PX = 1024

if not 0 <= IMAGE_INDEX < len(image_paths):
    raise ValueError(f'IMAGE_INDEX must be between 0 and {len(image_paths) - 1}.')
image_path = image_paths[IMAGE_INDEX]
image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
if image is None:
    raise OSError(f'Cannot read {image_path}')

preparation = None  # Release the previous run's large diagnostic snapshots.
preparation = prepare_board(image, normalized_size_px=NORMALIZED_SIZE_PX, diagnostics=True)
print(image_path.name, image.shape)
print(format_timing_summary(preparation))
if preparation.failure is not None:
    print(f'FAILED: {preparation.failure.stage}: {preparation.failure.message}')
elif preparation.board is not None:
    print(f'Color-derived rotation: {preparation.board.detector_rotation_deg:.2f} degrees (36-degree ambiguity)')


In [ ]:
figure = plot_board_preparation(preparation)
display(figure)
figure.clear()


## Inspect stage artifacts individually

Each step exposes `input_artifacts` and `output_artifacts`. Image artifacts may use different frames: the color mask is at detector resolution; ECC arrays are in the detector's refinement frame; `boundary_points` are source-image coordinates and `ellipse_inliers` selects them. Homographies are matrices, not display images. The following cells show optional ECC outputs and source boundary samples explicitly.


In [ ]:
for step in preparation.steps:
    print(f'\n{step.name}: {step.status.value}')
    for direction, artifacts in [('input', step.input_artifacts), ('output', step.output_artifacts)]:
        for name, array in artifacts.items():
            print(f'  {direction:6s} {name:20s} shape={array.shape} dtype={array.dtype}')

artifacts = preparation.steps[0].output_artifacts
ecc_names = [name for name in ('ecc_edges', 'ecc_template', 'ecc_mask') if name in artifacts]
if ecc_names:
    fig, axes = plt.subplots(1, len(ecc_names), figsize=(15, 5), squeeze=False)
    for ax, name in zip(axes.flat, ecc_names):
        ax.imshow(artifacts[name], cmap='gray')
        ax.set_title(name)
        ax.axis('off')
    plt.show()
    plt.close(fig)
else:
    print('No ECC image artifacts were returned for this run.')


In [ ]:
if 'boundary_points' in artifacts and 'ellipse_inliers' in artifacts:
    points = artifacts['boundary_points']
    inliers = artifacts['ellipse_inliers'].astype(bool)
    stride = max(1, int(np.ceil(max(image.shape[:2]) / 1400)))
    fig, ax = plt.subplots(figsize=(9, 10))
    ax.imshow(cv2.cvtColor(image[::stride, ::stride], cv2.COLOR_BGR2RGB), extent=(-0.5, image.shape[1] - 0.5, image.shape[0] - 0.5, -0.5))
    for selected, color, label in [(inliers, 'lime', 'ellipse inliers'), (~inliers, 'red', 'rejected samples')]:
        ax.scatter(points[selected, 0], points[selected, 1], s=12, c=color, label=label)
    ax.set_title('Boundary samples in source-image pixels')
    ax.legend()
    plt.show()
    plt.close(fig)
else:
    print('No boundary samples were returned for this run.')


## Optional resolution comparison (selected photo only)

Set `RUN_RESOLUTION_COMPARISON = True` to compare 512/1024/1536 px normalization. This reruns both real stages with diagnostics disabled, reports each timing, and displays rectified previews without retaining full-resolution snapshots. Single measurements are exploratory, not a statistical benchmark; board detection is independent of the normalization size.


In [ ]:
RUN_RESOLUTION_COMPARISON = False
if RUN_RESOLUTION_COMPARISON:
    sizes = (512, 1024, 1536)
    fig, axes = plt.subplots(1, len(sizes), figsize=(16, 6))
    for ax, size in zip(axes, sizes):
        candidate = prepare_board(image, normalized_size_px=size, diagnostics=False)
        print(f'\n{size} px\n{format_timing_summary(candidate)}')
        if candidate.failure is not None:
            print(f'FAILED: {candidate.failure.message}')
            ax.set_title(f'{size} px: FAILED')
        else:
            ax.imshow(cv2.cvtColor(candidate.board_view.image, cv2.COLOR_BGR2RGB))
            ax.set_title(f'{size} px')
        ax.axis('off')
    plt.show()
    plt.close(fig)
    del candidate


## Compact batch: final outputs only

Set `RUN_BATCH = True` and edit `BATCH_PATHS` to select any subset (remove `[:3]`
for the full directory). Run this cell again without repeating the intermediate
visualizations. Processing is sequential; pages retain small output thumbnails,
not full-resolution diagnostic histories. Each tile names the input, final
status and processing time. Failures are visible, not silently skipped.

To investigate a bad tile, copy its filename into this notebook's single-image
selection and rerun the detailed sections. Batch mode uses automatic calibration;
never reuse one image's manual points/tips on another image.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dartcv.notebook_batch import BatchOutput, image_batch_figures, read_batch_image

BATCH_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())

RUN_BATCH = False
BATCH_PATHS = sorted((BATCH_ROOT / "data/images/multi_dart").glob("*.jpg"))[:3]

from dartcv import prepare_board
from dartcv.color_dart_inspection import draw_board_geometry

def batch_output(path):
    prepared = prepare_board(read_batch_image(path), diagnostics=False)
    if prepared.failure is not None:
        return BatchOutput(None, f"FAILED: {prepared.failure.message}")
    view = prepared.board_view
    return BatchOutput(draw_board_geometry(view.image, view.model_to_view),
                       "Nominal board geometry; numbered orientation unverified")

if RUN_BATCH:
    for batch_figure in image_batch_figures(BATCH_PATHS, batch_output, page_size=6):
        display(batch_figure)
        plt.close(batch_figure)